# Phase F: Multi-Dataset Evaluation

Evaluate best checkpoint (and ablation checkpoints) across all 6 datasets:
ASVspoof 2019 LA, ITW, ASVspoof 2021 LA/DF, WaveFake, CodecFake+.

**Prerequisites:** Phase C (best checkpoint), Phase D (ablation checkpoints)


## 1. Environment Setup


In [ ]:
# Install dependencies
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    print('WARNING: No GPU detected')


## 1b. Copy GLADA Package to Local Runtime


In [ ]:
import shutil
import os
import sys

# Define paths
DRIVE_PKG = "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code/glada"
LOCAL_PKG = '/content/glada'

# Remove existing local package if it exists to ensure a fresh copy
if os.path.exists(LOCAL_PKG):
    shutil.rmtree(LOCAL_PKG)

# Copy the directory
try:
    shutil.copytree(DRIVE_PKG, LOCAL_PKG)
    print(f'Successfully copied package to {LOCAL_PKG}')
except Exception as e:
    print(f'Error copying package: {e}')

# Add to sys.path
if LOCAL_PKG not in sys.path:
    sys.path.insert(0, LOCAL_PKG)

# Verify
print(f'Package copied to {LOCAL_PKG}')
print(f"Contents of {LOCAL_PKG}: {os.listdir(LOCAL_PKG)}")


## 2. Package Imports

All model, dataset, and evaluation classes are imported from the `glada` package.
This guarantees checkpoint compatibility since the same code was used for training.


In [ ]:
import os, json, time, warnings, logging, gc
from pathlib import Path
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn

from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.models import GLADAModel
from glada.datasets import (
    ASVspoofEvalDataset,
    InTheWildDataset,
)
from glada.datasets.asvspoof2021 import ASVspoof2021Dataset
from glada.datasets.wavefake import WaveFakeDataset
from glada.datasets.codecfake import CodecFakeDataset
from glada.datasets.audio import load_audio
from glada.evaluation.evaluator import MultiDatasetEvaluator

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('All imports successful.')


## Dataset Paths and Extraction

Colab ephemeral storage is wiped between sessions. This cell verifies dataset availability and extracts from Google Drive archives if needed. All datasets are stored as compressed archives in Drive for persistence.

**Note**: ASVspoof 2021 LA and DF evaluation sets (Yamagishi et al., 2021) use a different protocol format than 2019 LA (Wang et al., 2020) — they provide only audio keys without ground truth labels during the challenge, but ground truth keys files are now publicly available.

### Dataset Sources

| Dataset | Archive Format | Approx. Size | Audio Format |
|---------|---------------|-------------|-------------|
| ASVspoof 2019 LA (Wang et al., 2020) | `.tar` | ~5 GB | FLAC, 16kHz |
| ASVspoof 2021 LA Eval (Yamagishi et al., 2021) | `.tar.gz` | ~2 GB | FLAC, various codecs |
| ASVspoof 2021 DF Eval (Yamagishi et al., 2021) | `.tar.gz` | ~3 GB | FLAC, deepfakes |
| In-the-Wild (M\u00fcller et al., 2024) | `.tar` | ~1.5 GB | Various formats |
| WaveFake (Frank & Sch\u00f6nherr, 2021) | `.tar.gz` | ~4 GB | WAV, 22050Hz (resampled to 16kHz) |

The extraction function handles both `.tar` and `.tar.gz` formats, skipping extraction if data is already present (idempotent). This is critical for Colab sessions where re-running cells should not re-extract gigabytes of data.

In [ ]:
# ============================================================================
# 2. Dataset Paths and Sequential Extract-Evaluate-Cleanup
# To fit within 70GB free-tier disk, we extract one dataset at a time,
# evaluate it, then delete the local copy before extracting the next.
# ============================================================================
import tarfile
import shutil

LOCAL_ROOT = '/content/data'
os.makedirs(LOCAL_ROOT, exist_ok=True)

DRIVE_BASE = '/content/drive/MyDrive/Jakob/Datasets'
CODECFAKE_BASE = '/content/drive/Shareddrives/Jakob-298-datasets/CodecFakePlus'

EVAL_BATCH_SIZE = 48  # Larger batch for eval (no gradients stored)

def extract_dataset(tar_path, extract_to, name):
    """Extract a tar/tar.gz archive to local storage."""
    if not os.path.exists(tar_path):
        print(f'  [{name}] Archive not found: {tar_path} — SKIPPING')
        return False
    os.makedirs(extract_to, exist_ok=True)
    if os.listdir(extract_to):
        print(f'  [{name}] Already extracted at {extract_to}')
        return True
    print(f'  [{name}] Extracting from {tar_path}...')
    mode = 'r:gz' if tar_path.endswith('.gz') else 'r:*'
    with tarfile.open(tar_path, mode) as tar:
        tar.extractall(path=extract_to)
    print(f'  [{name}] Done.')
    return True

def cleanup_dataset(local_dir, name):
    """Remove extracted dataset to free disk space."""
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
        print(f'  [{name}] Cleaned up {local_dir}')

def extract_multipart_dataset(part_pattern, extract_to, name):
    """Concatenate multipart tar.gz archives and extract.
    
    ASVspoof 2021 DF eval is split into parts on Drive due to size:
      ASVspoof2021_DF_eval_part_00, ASVspoof2021_DF_eval_part_01, ...
    These must be concatenated into a single tar.gz before extraction.
    """
    import glob
    parts = sorted(glob.glob(part_pattern))
    if not parts:
        print(f'  [{name}] No parts found matching {part_pattern} — SKIPPING')
        return False
    os.makedirs(extract_to, exist_ok=True)
    if os.listdir(extract_to):
        print(f'  [{name}] Already extracted at {extract_to}')
        return True
    
    concat_path = os.path.join(LOCAL_ROOT, f'{name}_concat.tar.gz')
    print(f'  [{name}] Concatenating {len(parts)} parts...')
    with open(concat_path, 'wb') as out_f:
        for p in parts:
            print(f'    {os.path.basename(p)}')
            with open(p, 'rb') as in_f:
                shutil.copyfileobj(in_f, out_f)
    
    print(f'  [{name}] Extracting concatenated archive...')
    with tarfile.open(concat_path, 'r:gz') as tar:
        tar.extractall(path=extract_to)
    
    # Remove concat file to free disk
    os.remove(concat_path)
    print(f'  [{name}] Done. Removed temp concat file.')
    return True

def find_keys_file(base_dir, track='LA'):
    """Auto-discover the keys/protocol file for ASVspoof 2021.
    
    The extraction structure varies by archive version. Common locations:
      - keys/LA/CM/trial_metadata.txt
      - ASVspoof2021_LA_eval/keys/LA/CM/trial_metadata.txt
      - LA/keys/CM/trial_metadata.txt
    """
    import glob
    patterns = [
        f'{base_dir}/**/trial_metadata.txt',
        f'{base_dir}/**/keys/{track}/CM/trial_metadata.txt',
    ]
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return matches[0]
    return None

def find_audio_dir(base_dir, track='LA'):
    """Auto-discover the audio directory for ASVspoof 2021."""
    import glob
    patterns = [
        f'{base_dir}/**/ASVspoof2021_{track}_eval/flac',
        f'{base_dir}/**/flac',
    ]
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return matches[0]
    return None

# Dataset registry: each entry defines how to extract, create dataloader, and cleanup.
# 'no_cleanup' prevents deletion — used when the next dataset shares the same archive.
DATASET_REGISTRY = [
    {
        'name': 'ITW',
        'archive': os.path.join(DRIVE_BASE, 'release_in_the_wild.tar'),
        'local_dir': os.path.join(LOCAL_ROOT, 'itw'),
        'make_loader': lambda: DataLoader(
            InTheWildDataset(
                '/content/data/itw/release_in_the_wild/meta.csv',
                '/content/data/itw/release_in_the_wild',
                max_len=config.max_audio_len, use_labels=True),
            batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    },
    {
        'name': '2019_LA_eval',
        'archive': os.path.join(DRIVE_BASE, 'LA.tar'),
        'local_dir': os.path.join(LOCAL_ROOT, 'asvspoof'),
        'no_cleanup': True,  # keep for 2019_LA_dev next
        'make_loader': lambda: DataLoader(
            ASVspoofEvalDataset(
                '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt',
                '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac',
                max_len=config.max_audio_len),
            batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    },
    {
        'name': '2019_LA_dev',
        'archive': None,  # reuses asvspoof extraction from above
        'local_dir': os.path.join(LOCAL_ROOT, 'asvspoof'),  # cleanup this dir after dev
        'make_loader': lambda: DataLoader(
            ASVspoofEvalDataset(
                '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt',
                '/content/data/asvspoof/LA/ASVspoof2019_LA_dev/flac',
                max_len=config.max_audio_len),
            batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    },
    {
        'name': '2021_LA',
        'archive': os.path.join(DRIVE_BASE, 'ASVspoof2021_LA_eval.tar.gz'),
        'local_dir': os.path.join(LOCAL_ROOT, 'asvspoof21_la'),
        'make_loader': 'auto_2021_LA',  # resolved at eval time via find_keys_file
    },
    {
        'name': '2021_DF',
        'archive': 'multipart',  # special: concatenate parts then extract
        'local_dir': os.path.join(LOCAL_ROOT, 'asvspoof21_df'),
        'make_loader': 'auto_2021_DF',  # resolved at eval time via find_keys_file
    },
    {
        'name': 'WaveFake',
        'archive': os.path.join(DRIVE_BASE, 'WaveFake.tar.gz'),
        'local_dir': os.path.join(LOCAL_ROOT, 'wavefake'),
        'make_loader': lambda: DataLoader(
            WaveFakeDataset(
                '/content/data/wavefake',
                max_len=config.max_audio_len),
            batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    },
]

print(f"Registered {len(DATASET_REGISTRY)} datasets for sequential evaluation.")
print("Datasets will be extracted, evaluated, and cleaned up one at a time.")


## Load Trained Model

Load a trained GLADA checkpoint. The checkpoint contains:
- `model`: model state dict (saved via `model.state_dict()`)
- `epoch`: training epoch at which the best EER was achieved
- `best_eer`: best validation EER during training
- `config`: full `GLADAConfig` as a dict for reproducibility

The `load_model` function reconstructs the config from the checkpoint's saved values, ensuring that architecture parameters (backbone, hidden_dim, num_groups, etc.) match exactly what was used during training. This is critical -- a mismatch in any of these causes a `state_dict` loading error.

In [ ]:
# ============================================================================
# Load trained GLADA model from checkpoint
# Checkpoint format: {'model': state_dict, 'epoch': int, 'best_eer': float,
#                      'config': dict}  (saved by training loop)
# ============================================================================

def load_model(checkpoint_path, backbone='wavlm-large'):
    """Load trained GLADA model from checkpoint.
    
    Args:
        checkpoint_path: Path to .pt file with model state_dict + config
        backbone: Backbone key (used as fallback if config not in checkpoint)
    
    Returns:
        model: GLADAModel in eval mode
        config: GLADAConfig reconstructed from checkpoint
    """
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    # Reconstruct config from checkpoint (ensures architecture match)
    saved_config = ckpt.get('config', {})
    config = GLADAConfig(backbone=backbone)
    # Override with saved config values to match training exactly
    for k, v in saved_config.items():
        if hasattr(config, k):
            setattr(config, k, v)
    config.__post_init__()  # Recompute derived values (hf_model_id, hidden_dim, etc.)

    # Build model and load weights
    model = GLADAModel(config)
    model.load_state_dict(ckpt['model'])
    model.eval()  # Set to eval mode (deterministic VIB, no dropout)

    # Print checkpoint info
    epoch = ckpt.get('epoch', '?')
    best_eer = ckpt.get('best_eer', '?')
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Loaded from epoch {epoch}, best EER: {best_eer}")
    print(f"Backbone: {config.backbone} ({config.hf_model_id})")
    print(f"Parameters: {n_params:.1f}M")
    print(f"Components: GLADA={config.use_glada}, VIB={config.use_vib}, "
          f"CKA={config.use_cka_loss}, Speaker={config.use_speaker_adversary}")
    return model, config


# ============================================================================
# CONFIGURE CHECKPOINT PATH HERE
# Update the path below to point to your trained model checkpoint.
# Expected location: /content/drive/MyDrive/Jakob/Experiments/<experiment>/<run_id>/best_model.pt
# ============================================================================

CHECKPOINT = '/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed42/YYYYMMDD_HHMMSS/best_model.pt'  # <-- UPDATE THIS
BACKBONE = 'wavlm-large'  # Must match training backbone

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model, config = load_model(CHECKPOINT, BACKBONE)
model = model.to(device)

### Legacy Model Loading (DANN-IB from Survey Paper)

The preliminary experiments in Section VIII of the survey paper used `DomainAdaptationModel`
from `domain_adaptation_e2e.py` — a simpler architecture than GLADA (frozen SSL backbone +
linear classifier + optional domain discriminator). To re-evaluate those checkpoints on all
5 standard datasets, we need to reconstruct the legacy model class.

**When to use this**: Set `USE_LEGACY = True` below if evaluating old DANN-IB/CAAN/MMD/CORAL
checkpoints from `/content/drive/MyDrive/Jakob/Experiments/` (configs matching the survey paper
Tables III-V).

**Configs to re-evaluate** (from survey paper):
- WavLM-Base-Plus + DANN-IB (seeds 42, 123, 456)
- WavLM-Base-Plus + CAAN (seeds 42, 123, 456)
- WavLM-Large + CAAN (seeds 42, 123, 456)
- WavLM-Large + DANN-IB (seeds 42, 123, 456)
- XLS-R 300M + DANN-IB (seeds 42, 123, 456)


In [ ]:
# ============================================================================
# LEGACY MODEL LOADING (DANN-IB / Survey Paper Experiments)
# ============================================================================
# The old DomainAdaptationModel uses a frozen SSL backbone with mean pooling
# and a linear classifier head. Much simpler than GLADA.

USE_LEGACY = False  # Set True to evaluate old DANN-IB checkpoints

class LegacyGradientReversal(torch.autograd.Function):
    """GRL from domain_adaptation_e2e.py."""
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None


class LegacyDomainAdaptationModel(nn.Module):
    """Reconstructed from domain_adaptation_e2e.py for checkpoint loading.
    Architecture: frozen SSL -> mean pool -> classifier head.
    Optional: domain discriminator with GRL."""

    def __init__(self, model_name='microsoft/wavlm-base-plus', hidden_size=768,
                 has_domain_disc=True):
        super().__init__()
        self.wavlm = AutoModel.from_pretrained(model_name)
        self.wavlm.feature_extractor._freeze_parameters()
        for p in self.wavlm.parameters():
            p.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 2)
        )
        self.domain_disc = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 2)
        ) if has_domain_disc else None
        self.grl_alpha = 0.0

    def forward(self, input_values, attention_mask=None):
        features = self.wavlm(
            input_values, attention_mask=attention_mask
        ).last_hidden_state.mean(1)
        result = {
            'spoof_logits': self.classifier(features),
            'features': features
        }
        if self.domain_disc is not None:
            result['domain_logits'] = self.domain_disc(
                LegacyGradientReversal.apply(features, self.grl_alpha)
            )
        return result


# --- Legacy model name mapping ---
LEGACY_BACKBONE_MAP = {
    'wavlm-base-plus': ('microsoft/wavlm-base-plus', 768),
    'wavlm-large': ('microsoft/wavlm-large', 1024),
    'xlsr-300m': ('facebook/wav2vec2-xls-r-300m', 1024),
    'wav2vec2-base': ('facebook/wav2vec2-base-960h', 768),
    'wav2vec2-large': ('facebook/wav2vec2-large-960h', 1024),
}


def load_legacy_model(checkpoint_path, backbone='wavlm-base-plus'):
    """Load a DomainAdaptationModel checkpoint from old experiments.

    The old checkpoints save state_dict with keys like 'wavlm.encoder...',
    'classifier.0.weight', 'domain_disc.0.weight'. The key 'task_logits'
    in the old model maps to 'spoof_logits' in evaluation code.
    """
    model_name, hidden = LEGACY_BACKBONE_MAP[backbone]
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    # Check if domain_disc keys exist in state_dict
    state = ckpt['model']
    has_disc = any('domain_disc' in k for k in state.keys())

    model = LegacyDomainAdaptationModel(
        model_name=model_name, hidden_size=hidden, has_domain_disc=has_disc
    )
    model.load_state_dict(state)
    model.eval()

    epoch = ckpt.get('epoch', '?')
    best_eer = ckpt.get('best_eer', ckpt.get('eer', '?'))
    print(f'Legacy model loaded: {backbone} (epoch {epoch}, EER {best_eer})')
    print(f'  Domain disc: {has_disc}')
    return model


# --- Override model if using legacy ---
if USE_LEGACY:
    LEGACY_CHECKPOINT = '/content/drive/MyDrive/Jakob/Experiments/dann_ib_wavlm_bp_seed42/best_model.pt'  # Edit path
    LEGACY_BACKBONE = 'wavlm-base-plus'  # Edit backbone
    model = load_legacy_model(LEGACY_CHECKPOINT, LEGACY_BACKBONE)
    model = model.to(device)
    print('Using LEGACY model for evaluation')


## Multi-Dataset Evaluation

Using `MultiDatasetEvaluator` from the `glada.evaluation` package.


## Sequential Evaluation

Extracts one dataset at a time, evaluates, then deletes to stay within 70GB disk.

In [ ]:
# ============================================================================
# Sequential Evaluation: extract -> eval -> cleanup -> next
# Fits within 70GB free-tier disk by only keeping one dataset at a time.
# ============================================================================

# Determine AMP dtype based on GPU
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0).lower()
    amp_dtype = torch.bfloat16 if 'a100' in gpu_name or 'h100' in gpu_name else torch.float16
else:
    amp_dtype = torch.float32
print(f"AMP dtype: {amp_dtype}")

evaluator = MultiDatasetEvaluator(model, device, amp_dtype=amp_dtype)

all_results = {}
print("=" * 70)
print("SEQUENTIAL EVALUATION (extract -> eval -> cleanup)")
print("=" * 70)

for ds in DATASET_REGISTRY:
    name = ds['name']
    print(f"\n{'─' * 50}")
    print(f"Dataset: {name}")
    print(f"{'─' * 50}")

    # 1. Extract (if this dataset has its own archive)
    if ds.get('archive'):
        if ds['archive'] == 'multipart':
            # ASVspoof 2021 DF: concatenate split archives then extract
            ok = extract_multipart_dataset(
                '/content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF/ASVspoof2021_DF_eval_part_*',
                ds['local_dir'], name)
        else:
            ok = extract_dataset(ds['archive'], ds['local_dir'], name)
        if not ok:
            print(f"  SKIPPING {name} (archive not found)")
            continue

    # 2. Create dataloader
    try:
        maker = ds['make_loader']
        if maker == 'auto_2021_LA':
            keys = find_keys_file(ds['local_dir'], 'LA')
            audio = find_audio_dir(ds['local_dir'], 'LA')
            if not keys or not audio:
                import subprocess
                tree = subprocess.run(['find', ds['local_dir'], '-maxdepth', '3', '-type', 'f'],
                                      capture_output=True, text=True)
                print(f"  Keys file: {keys}")
                print(f"  Audio dir: {audio}")
                print(f"  Extracted files (first 20):")
                for line in tree.stdout.strip().split('\n')[:20]:
                    print(f"    {line}")
                raise FileNotFoundError(f"Could not find keys or audio for 2021 LA")
            print(f"  Auto-discovered keys: {keys}")
            print(f"  Auto-discovered audio: {audio}")
            loader = DataLoader(
                ASVspoof2021Dataset(keys, audio, max_len=config.max_audio_len),
                batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
        elif maker == 'auto_2021_DF':
            keys = find_keys_file(ds['local_dir'], 'DF')
            audio = find_audio_dir(ds['local_dir'], 'DF')
            if not keys or not audio:
                import subprocess
                tree = subprocess.run(['find', ds['local_dir'], '-maxdepth', '3', '-type', 'f'],
                                      capture_output=True, text=True)
                print(f"  Keys file: {keys}")
                print(f"  Audio dir: {audio}")
                print(f"  Extracted files (first 20):")
                for line in tree.stdout.strip().split('\n')[:20]:
                    print(f"    {line}")
                raise FileNotFoundError(f"Could not find keys or audio for 2021 DF")
            print(f"  Auto-discovered keys: {keys}")
            print(f"  Auto-discovered audio: {audio}")
            loader = DataLoader(
                ASVspoof2021Dataset(keys, audio, max_len=config.max_audio_len),
                batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
        else:
            loader = maker()
        print(f"  Loaded {len(loader.dataset)} samples")
    except Exception as e:
        print(f"  ERROR creating dataloader for {name}: {e}")
        if ds.get('local_dir') and not ds.get('no_cleanup'):
            cleanup_dataset(ds['local_dir'], name)
        continue

    # 3. Evaluate
    result = evaluator.evaluate_dataset(loader, name)
    all_results[name] = result
    print(f"  EER: {result['eer']:.2f}% [{result.get('ci_low', 0):.2f}–{result.get('ci_high', 0):.2f}%]")

    # 4. Cleanup to free disk (unless no_cleanup is set)
    del loader
    torch.cuda.empty_cache()
    if not ds.get('no_cleanup') and ds.get('local_dir'):
        cleanup_dataset(ds['local_dir'], name)

    # Show remaining disk
    import subprocess
    df_out = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
    print(f"  Disk: {df_out.stdout.strip().split(chr(10))[-1]}")

# --- Display results ---
print("\n" + "=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)
md_table = MultiDatasetEvaluator.results_to_markdown(all_results)
print(md_table)
latex_table = MultiDatasetEvaluator.results_to_latex(all_results)
print("\n### LaTeX Table")
print(latex_table)

# --- Save to Drive ---
results_save_path = os.path.join(
    config.drive_log_dir,
    f'eval_{config.backbone}_results.json'
)
os.makedirs(os.path.dirname(results_save_path), exist_ok=True)
with open(results_save_path, 'w') as f:
    json.dump({
        'backbone': config.backbone,
        'results': all_results,
    }, f, indent=2, default=str)
print(f"\nResults saved to: {results_save_path}")


## Multi-Model Comparison

Compare multiple trained checkpoints (different backbones, seeds, or ablation configurations) across all evaluation datasets. This produces the main results table for the thesis.

**Expected comparisons**:
- **Backbone comparison**: WavLM-Large vs XLSR-300M vs Data2Vec-Large vs Wav2Vec2-Robust
- **Seed stability**: Same backbone across seeds {42, 123, 2026} to assess variance
- **Ablation variants**: Full GLADA vs components removed (no GLADA, no VIB, no CKA, no speaker adversary)

### Why Multi-Model Comparison Matters

Single-model results can be misleading due to seed sensitivity (Rule 4 requires multi-seed experiments with statistical reporting). The comparison table shows:
1. Whether GLADA consistently improves over baselines across backbones
2. Whether the improvement is robust to random seed variation
3. Which components contribute most to performance (ablation analysis)

The results dictionary maps `model_name -> {dataset_name -> eval_metrics}` and is saved to JSON for later analysis.

In [ ]:
# ============================================================================
# Multi-Model Comparison: evaluate multiple checkpoints on all datasets
# Produces the main results table for the thesis (Table X in manuscript)
# ============================================================================

# --- CONFIGURE CHECKPOINTS HERE ---
# Map from display name -> (checkpoint_path, backbone_key)
# Update these paths to match your trained checkpoints.
CHECKPOINTS = {
    # Backbone comparison (same config, different SSL models)
    # 'WavLM-Large':      ('/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed42/.../best_model.pt', 'wavlm-large'),
    # 'XLSR-300M':        ('/content/drive/MyDrive/Jakob/Experiments/glada_xlsr-300m_seed42/.../best_model.pt', 'xlsr-300m'),
    # 'Data2Vec-Large':   ('/content/drive/MyDrive/Jakob/Experiments/glada_data2vec-large_seed42/.../best_model.pt', 'data2vec-large'),
    # 'Wav2Vec2-Robust':  ('/content/drive/MyDrive/Jakob/Experiments/glada_wav2vec2-robust_seed42/.../best_model.pt', 'wav2vec2-robust'),

    # Seed stability (same backbone, different seeds)
    # 'WavLM-Large (s42)':   ('/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed42/.../best_model.pt', 'wavlm-large'),
    # 'WavLM-Large (s123)':  ('/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed123/.../best_model.pt', 'wavlm-large'),
    # 'WavLM-Large (s2026)': ('/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed2026/.../best_model.pt', 'wavlm-large'),

    # Ablation variants
    # 'Full GLADA':       ('/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed42/.../best_model.pt', 'wavlm-large'),
    # 'No GLADA':         ('/content/drive/MyDrive/Jakob/Experiments/ablation_no_glada/.../best_model.pt', 'wavlm-large'),
    # 'No VIB':           ('/content/drive/MyDrive/Jakob/Experiments/ablation_no_vib/.../best_model.pt', 'wavlm-large'),
    # 'No CKA':           ('/content/drive/MyDrive/Jakob/Experiments/ablation_no_cka/.../best_model.pt', 'wavlm-large'),
    # 'No Speaker GRL':   ('/content/drive/MyDrive/Jakob/Experiments/ablation_no_speaker/.../best_model.pt', 'wavlm-large'),
}

if len(CHECKPOINTS) == 0:
    print("No checkpoints configured for multi-model comparison.")
    print("Uncomment and update the checkpoint paths above, then re-run this cell.")
else:
    comparison_results = {}  # {model_name: {dataset_name: result_dict}}

    for model_name, (ckpt_path, backbone_key) in CHECKPOINTS.items():
        print(f"\n{'='*70}")
        print(f"Evaluating: {model_name}")
        print(f"{'='*70}")

        try:
            # Load model
            m, cfg = load_model(ckpt_path, backbone_key)
            m = m.to(device)

            # Evaluate on all datasets
            ev = MultiDatasetEvaluator(m, device, amp_dtype=amp_dtype)
            comparison_results[model_name] = ev.evaluate_all(eval_datasets)

            # Free GPU memory for next model
            del m, ev
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"  ERROR loading {model_name}: {e}")
            comparison_results[model_name] = {'error': str(e)}

    # --- Print comparison table ---
    print("\n" + "="*70)
    print("MULTI-MODEL COMPARISON")
    print("="*70)

    # Collect all dataset names across all models
    all_dataset_names = set()
    for model_name, res in comparison_results.items():
        if 'error' not in res:
            all_dataset_names.update(res.keys())
    all_dataset_names = sorted(all_dataset_names)

    # Markdown comparison table
    header = "| Model | " + " | ".join([f"{d} EER (%)" for d in all_dataset_names]) + " |"
    separator = "|" + "|".join(["---"] * (1 + len(all_dataset_names))) + "|"
    print(header)
    print(separator)

    for model_name, res in comparison_results.items():
        if 'error' in res:
            row = f"| {model_name} | " + " | ".join(["ERROR"] * len(all_dataset_names)) + " |"
        else:
            cells = []
            for d in all_dataset_names:
                if d in res:
                    r = res[d]
                    cells.append(f"{r['eer']:.2f} [{r.get('ci_low', r['eer']):.2f}, {r.get('ci_high', r['eer']):.2f}]")
                else:
                    cells.append("N/A")
            row = f"| {model_name} | " + " | ".join(cells) + " |"
        print(row)

    # --- LaTeX comparison table ---
    print("\n### LaTeX Comparison Table")
    n_cols = 1 + len(all_dataset_names)
    col_spec = "l" + "c" * len(all_dataset_names)
    print(r"\begin{table}[h]")
    print(r"\centering")
    print(r"\caption{Multi-model comparison. EER (\%) with 95\% bootstrap CI.}")
    print(r"\label{tab:multi_model}")
    print(f"\\begin{{tabular}}{{{col_spec}}}")
    print(r"\toprule")
    print("Model & " + " & ".join(all_dataset_names) + r" \\")
    print(r"\midrule")
    for model_name, res in comparison_results.items():
        if 'error' in res:
            row_cells = ["--"] * len(all_dataset_names)
        else:
            row_cells = []
            for d in all_dataset_names:
                if d in res:
                    r = res[d]
                    row_cells.append(f"{r['eer']:.2f}")
                else:
                    row_cells.append("--")
        print(f"{model_name} & " + " & ".join(row_cells) + r" \\")
    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table}")

    # --- Save comparison results to JSON ---
    comparison_save_path = os.path.join(
        config.drive_log_dir, 'multi_model_comparison.json'
    )
    with open(comparison_save_path, 'w') as f:
        json.dump(comparison_results, f, indent=2, default=str)
    print(f"\nComparison results saved to: {comparison_save_path}")

## SOTA Comparison

Published results from the literature for comparison. Values are EER (%) on standard evaluation sets.

**Sources**: IB-CAAN (Huang et al., 2025), HierCon (Liang et al., 2026), MultiConv+CKA (Tran et al., 2025), AASIST (Jung et al., 2022), RawNet2 (Tak et al., 2021).

### Notes on Comparison Fairness

- All systems listed train on ASVspoof 2019 LA and evaluate on ITW (Muller et al., 2022)
- Some systems (IB-CAAN, MultiConv) use additional augmentation or ensembles
- Our results use the same evaluation protocol for direct comparison
- AASIST and RawNet2 use fixed-architecture backends (no SSL backbone)
- SSL-based methods (HierCon, IB-CAAN, GLADA) generally outperform fixed-architecture methods on cross-domain benchmarks

In [ ]:
# ============================================================================
# SOTA Comparison Table
# Published EER (%) results from the literature vs our GLADA results.
# All methods train on ASVspoof 2019 LA; ITW is the cross-domain benchmark.
# ============================================================================

# --- Published SOTA results ---
# Format: (method, backbone, ITW EER %, ASVspoof 2019 LA EER %, reference)
SOTA_RESULTS = [
    # Method              Backbone             ITW EER   LA EER   Reference
    ('RawNet2',          'RawNet2 (fixed)',     31.20,    5.13,   'Tak et al., Interspeech 2021'),
    ('AASIST',           'AASIST (fixed)',      24.70,    0.83,   'Jung et al., ICASSP 2022'),
    ('AASIST + SSL',     'Wav2Vec2-XLSR',      10.80,    0.90,   'Tak et al., Interspeech 2022'),
    ('HierCon',          'XLS-R-300M',           6.87,    None,   'Liang et al., arXiv 2602.01032, 2026'),
    ('MultiConv+CKA',    'WavLM-Large',          4.44,    None,   'Tran et al., ACM MM 2025'),
    ('IB-CAAN',          'WavLM-Large',          4.93,    None,   'Huang et al., arXiv 2509.23618, 2025'),
]

# --- Our GLADA results (populated from evaluation above) ---
# If multi-model comparison was run, use those results;
# otherwise fall back to the single-model evaluation.
our_results = []

if 'comparison_results' in dir() and len(comparison_results) > 0:
    # Use multi-model comparison results
    for model_name, res in comparison_results.items():
        if 'error' in res:
            continue
        itw_eer = res.get('ITW', {}).get('eer', None)
        la_eer = res.get('2019_LA_eval', {}).get('eer', None)
        our_results.append((
            f'GLADA ({model_name})',
            model_name,
            itw_eer,
            la_eer,
            'Ours'
        ))
elif 'all_results' in dir() and len(all_results) > 0:
    # Use single-model evaluation results
    itw_eer = all_results.get('ITW', {}).get('eer', None)
    la_eer = all_results.get('2019_LA_eval', {}).get('eer', None)
    our_results.append((
        f'GLADA (ours)',
        config.backbone,
        itw_eer,
        la_eer,
        'Ours'
    ))

# --- Markdown SOTA table ---
print("### SOTA Comparison (Markdown)")
print("| Method | Backbone | ITW EER (%) | LA EER (%) | Reference |")
print("|--------|----------|-------------|------------|-----------|")
for method, backbone, itw, la, ref in SOTA_RESULTS:
    itw_str = f"{itw:.2f}" if itw is not None else "--"
    la_str = f"{la:.2f}" if la is not None else "--"
    print(f"| {method} | {backbone} | {itw_str} | {la_str} | {ref} |")
# Our results (highlighted)
for method, backbone, itw, la, ref in our_results:
    itw_str = f"**{itw:.2f}**" if itw is not None else "--"
    la_str = f"**{la:.2f}**" if la is not None else "--"
    print(f"| {method} | {backbone} | {itw_str} | {la_str} | {ref} |")

# --- LaTeX SOTA table ---
print("\n### SOTA Comparison (LaTeX -- copy to thesis)")
print(r"\begin{table}[t]")
print(r"\centering")
print(r"\caption{Comparison with published methods. EER (\%) on ASVspoof 2019 LA eval and In-the-Wild (ITW) datasets. All methods train on ASVspoof 2019 LA. Lower is better.}")
print(r"\label{tab:sota_comparison}")
print(r"\begin{tabular}{llcc}")
print(r"\toprule")
print(r"Method & Backbone & ITW EER (\%) & LA EER (\%) \\")
print(r"\midrule")
for method, backbone, itw, la, ref in SOTA_RESULTS:
    itw_str = f"{itw:.2f}" if itw is not None else "--"
    la_str = f"{la:.2f}" if la is not None else "--"
    print(f"{method}~\\cite{{{ref.split(',')[0].lower().replace(' ', '_')}}} & {backbone} & {itw_str} & {la_str} \\\\")
print(r"\midrule")
for method, backbone, itw, la, ref in our_results:
    itw_str = f"\\textbf{{{itw:.2f}}}" if itw is not None else "--"
    la_str = f"\\textbf{{{la:.2f}}}" if la is not None else "--"
    print(f"{method} & {backbone} & {itw_str} & {la_str} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

# --- Save complete SOTA comparison to Drive ---
sota_save_path = os.path.join(config.drive_log_dir, 'sota_comparison.json')
os.makedirs(os.path.dirname(sota_save_path), exist_ok=True)
sota_data = {
    'published': [
        {'method': m, 'backbone': b, 'itw_eer': i, 'la_eer': l, 'reference': r}
        for m, b, i, l, r in SOTA_RESULTS
    ],
    'ours': [
        {'method': m, 'backbone': b, 'itw_eer': i, 'la_eer': l, 'reference': r}
        for m, b, i, l, r in our_results
    ],
}
with open(sota_save_path, 'w') as f:
    json.dump(sota_data, f, indent=2, default=str)
print(f"\nSOTA comparison saved to: {sota_save_path}")